In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'yt-dlp>=2024.11.18',
    'huggingface-hub>=0.26.0',
    'python-dotenv>=1.0.0',
    'pyyaml>=6.0',
    'requests>=2.32.0',
    'soundfile>=0.12.1',
    'numpy>=1.26.0',
    'pyloudnorm>=0.1.1',
    'pyarrow>=16.0.0',
], check=True)
subprocess.run(['apt-get', 'install', '-qq', '-y', 'ffmpeg'], check=True)
try:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '*.whl'], check=True)
except Exception:
    pass

In [ ]:
import os, sys
sys.path.insert(0, '/kaggle/input/datasets/mirza176528/s2s-pipline-v2-0-2')

from shared.secrets import load_secrets
from shared.cf_client import CFClient
from shared.workflow_kernel import WorkflowKernel
from shared.repo_router import RepoRouter

import yaml
from pathlib import Path

CONFIG_DIR = Path('/kaggle/input/urdu-asr-pipelines/config')
WORK_DIR   = Path('/kaggle/working')
WORK_DIR.mkdir(parents=True, exist_ok=True)

RUN_ID       = 'run_20260507_001'
SESSION_ID   = 'cpu_collect_01'
SESSION_TYPE = 'cpu_collect'
SHARD_KEY    = 'cpu'

SECRETS = load_secrets(require_gemini=False)

with open(CONFIG_DIR / 'hf_repos.yaml') as f:
    repos_cfg = yaml.safe_load(f)

STAGE0_REPO   = repos_cfg['repos']['stage0_codec']['repo_id']
OVERFLOW_REPO = repos_cfg['repos']['overflow']['repo_id']

kernel = WorkflowKernel(
    run_id           = RUN_ID,
    session_id       = SESSION_ID,
    session_type     = SESSION_TYPE,
    shard_key        = SHARD_KEY,
    cf_worker_url    = SECRETS['CF_WORKER_URL'],
    cf_worker_secret = SECRETS['CF_WORKER_SECRET'],
    gpu_type         = None,
    vram_limit_gb    = 0.0,
    session_max_hours = 8.5,
)
kernel.start()
print(f'[session] {SESSION_ID} started — run={RUN_ID}')

In [ ]:
repo_router = RepoRouter(STAGE0_REPO, OVERFLOW_REPO)

stages_to_run = ['p1a', 'p1b', 'p1e']

for stage in stages_to_run:
    if kernel.session_expiring:
        print(f'[session] expiring — skipping {stage}')
        break
    kernel.check_session_time()

    started_at = kernel.log_stage_start(stage)
    try:
        if stage == 'p1a':
            exec(open('/kaggle/input/urdu-asr-pipelines/pipeline_1_collect/p1a_discover.ipynb').read())
        elif stage == 'p1b':
            exec(open('/kaggle/input/urdu-asr-pipelines/pipeline_1_collect/p1b_download.ipynb').read())
        elif stage == 'p1e':
            exec(open('/kaggle/input/urdu-asr-pipelines/pipeline_1_collect/p1e_upload.ipynb').read())
        kernel.log_stage_end(stage, started_at)
        print(f'[session] {stage} completed')
    except Exception as e:
        kernel.log_stage_end(stage, started_at, error=str(e))
        print(f'[session] {stage} failed: {e}')
        raise

kernel.stop()
print('[session] cpu_collect session complete')